# EXP-031 · Hierarchical cultural-ranking Qwen 7B

One complete fold-3 exploratory run: fitting-only language prior, shared image sentiment and low-rank image-language interaction, plus confidence-weighted ranking and eight-way vote supervision. Select A100 or L4, authorize below and choose Run all once. The same notebook executes a real full-painting CUDA feasibility check and resumes verified Drive checkpoints. No Development or sealed confirmation scoring.


In [ ]:
# OWNER_CONFIGURATION
AUTHORIZE_EXPERIMENT = False
EXECUTION_SCOPE = "RESEARCH_ONLY_CV"  # One exploratory fold; six epochs maximum.
RESOURCE_ELIGIBILITY_STATUS = "PINNED_APACHE_2_MODEL_RESEARCH_ONLY"
AUTHORIZE_GUARDED_CONFIRMATION = False
REQUIRE_SUBMISSION_ZIP = False
EVAL_INPUT_CSV = None
EVAL_DATA_DIR = None
MAX_GPU_HOURS = 12.0
MAX_TOTAL_WALL_HOURS = 12.0
AUTO_BROWSER_DOWNLOAD = True
AUTO_RELEASE_COLAB = True
HANDOFF_PATH = "/content/drive/MyDrive/exp023_handoff.zip"
RUN_ROOT = "/content/drive/MyDrive/exp031"
SOURCE = "df08f383ac4333953e3feda260dd7f529fc1cedc"


In [ ]:
from datetime import datetime, timezone
import json
import math
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
from uuid import uuid4

REPOSITORY = "https://github.com/bqa100507-spec/SemEval-2027-Task-12---ArtEmotion-28-Subtask-3---Image-Only-Cross-Cultural-Sentiment-Prediction.git"


def run_cloud():
    if AUTHORIZE_EXPERIMENT is not True:
        print("GPU authorization is off. Review the configuration before Run all.")
        return

    from google.colab import drive, files, runtime
    started = time.time()
    attempt = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
    root = Path(RUN_ROOT)
    destination = root / "attempts" / attempt
    local = Path(tempfile.mkdtemp(prefix="exp031-", dir="/content"))
    output = local / "report"
    output.mkdir()
    source = local / "source"
    finalize = None
    stage = "configuration"
    report = {"status": "BLOCKED", "attempt": attempt, "source": SOURCE, "scope": EXECUTION_SCOPE}
    env = dict(os.environ, CUBLAS_WORKSPACE_CONFIG=":4096:8", HF_HUB_DISABLE_XET="1",
               UV_PYTHON_INSTALL_DIR=str(local / "python"), UV_CACHE_DIR=str(local / "uv_cache"),
               OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", OMP_THREAD_LIMIT="1", MKL_NUM_THREADS="1")
    env.pop("PYTHONPATH", None)

    def run(command, name, *, cwd=None):
        nonlocal stage
        stage = name
        remaining = started + hours * 3600 - time.time() - 150
        if remaining <= 0:
            raise TimeoutError("Budget exhausted before " + name)
        log = output / (name + ".log")
        print(f"{name} — {remaining / 3600:.2f}h remaining", flush=True)
        if name == "train":
            with log.open("w") as handle:
                child = subprocess.Popen(command, cwd=cwd, env=env, stdout=handle, stderr=subprocess.STDOUT)
                try:
                    with log.open() as reader:
                        while child.poll() is None:
                            chunk = reader.read()
                            if chunk:
                                print(chunk, end="", flush=True)
                            if time.time() >= started + hours * 3600 - 150:
                                raise TimeoutError("Worker deadline. Published epochs remain on Drive.")
                            time.sleep(3)
                        print(reader.read(), end="", flush=True)
                    if child.returncode:
                        raise subprocess.CalledProcessError(child.returncode, command)
                finally:
                    if child.poll() is None:
                        child.terminate()
                        try:
                            child.wait(timeout=10)
                        except subprocess.TimeoutExpired:
                            child.kill()
                            child.wait()
        else:
            with log.open("w") as handle:
                subprocess.run(command, cwd=cwd, env=env, check=True, timeout=remaining,
                               stdout=handle, stderr=subprocess.STDOUT)

    try:
        if (EXECUTION_SCOPE != "RESEARCH_ONLY_CV" or AUTHORIZE_GUARDED_CONFIRMATION or REQUIRE_SUBMISSION_ZIP
                or EVAL_INPUT_CSV is not None or EVAL_DATA_DIR is not None
                or RESOURCE_ELIGIBILITY_STATUS != "PINNED_APACHE_2_MODEL_RESEARCH_ONLY"):
            raise ValueError("EXP-031 allows only the training-data fold-3 research run.")
        if any(not math.isfinite(x) or not 0 < x <= 12 for x in (MAX_GPU_HOURS, MAX_TOTAL_WALL_HOURS)):
            raise ValueError("Choose positive finite GPU and wall limits of at most 12 hours.")
        hours = min(MAX_GPU_HOURS, MAX_TOTAL_WALL_HOURS)
        stage = "drive"
        drive.mount("/content/drive")
        destination.mkdir(parents=True, exist_ok=False)
        run(["git", "clone", "--no-checkout", REPOSITORY, str(source)], "clone")
        run(["git", "checkout", "--detach", SOURCE], "checkout", cwd=source)
        sys.path.insert(0, str(source / "src"))
        from st3.experiment_workflow import finalize

        stage = "input"
        if not Path(HANDOFF_PATH).is_file():
            raise FileNotFoundError("Set HANDOFF_PATH to the original TASK-034 exp023_handoff.zip.")
        run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"], "gpu")
        gpu = (output / "gpu.log").read_text()
        print(gpu)
        if not ("A100" in gpu or "L4" in gpu):
            raise RuntimeError("Select an A100 or L4 hardware accelerator.")

        uv = [sys.executable, "-m", "uv"]
        run([sys.executable, "-m", "pip", "install", "uv==0.7.13"], "uv")
        run([*uv, "python", "install", "3.12.10"], "python")
        run([*uv, "venv", "--python", "3.12.10", str(local / "venv")], "venv")
        python = str(local / "venv/bin/python")
        run([*uv, "pip", "sync", "--python", python, "--extra-index-url", "https://download.pytorch.org/whl/cu126",
             "--index-strategy", "unsafe-best-match", "requirements-exp026.lock.txt"], "dependencies", cwd=source)
        run([*uv, "pip", "check", "--python", python], "dependency_check")
        run([python, "-m", "pytest", "-q", "tests/test_exp031.py", "tests/test_exp030.py", "tests/test_experiment_workflow.py"],
            "cpu_tests", cwd=source)
        run([python, "-u", "scripts/run_exp031.py", "--handoff", HANDOFF_PATH, "--source", SOURCE,
             "--work", str(local / "work"), "--output", str(output), "--checkpoints", str(root / "checkpoints"),
             "--hours", str(hours), "--started", str(started), "--authorize-research", "EXP031"], "train", cwd=source)
        for filename in ("results.json", "failure.json"):
            path = output / filename
            if path.exists():
                report.update(json.loads(path.read_text()))
        print("Drive attempt:", destination)
    except Exception as error:
        for filename in ("results.json", "failure.json"):
            path = output / filename
            if path.exists():
                report.update(json.loads(path.read_text()))
        report.update(stage=stage, attempt_error=f"{type(error).__name__}: {error}",
                      status="STOPPED_BUDGET" if isinstance(error, (TimeoutError, subprocess.TimeoutExpired)) else "BLOCKED")
        if (output / (stage + ".log")).exists():
            print((output / (stage + ".log")).read_text(errors="replace")[-3500:])
        raise
    finally:
        report["allocation_elapsed_before_export_seconds"] = time.time() - started
        if finalize is not None:
            finalize(output, destination, report, platform="colab", download=files.download,
                     release=runtime.unassign, auto_download=AUTO_BROWSER_DOWNLOAD, auto_release=AUTO_RELEASE_COLAB)
        else:
            try:
                if destination.is_dir():
                    (output / "bootstrap_status.json").write_text(json.dumps(report, indent=2))
                    local_zip = Path(shutil.make_archive(str(local / "outputs"), "zip", output))
                    target = destination / "outputs.zip"
                    shutil.copyfile(local_zip, target)
                    import hashlib
                    digest = hashlib.sha256(local_zip.read_bytes()).hexdigest()
                    if hashlib.sha256(target.read_bytes()).hexdigest() != digest:
                        raise OSError("Bootstrap archive Drive read-back mismatch")
                    (destination / "outputs.sha256").write_text(digest + "  outputs.zip\n")
                    if AUTO_BROWSER_DOWNLOAD:
                        files.download(str(target))
            finally:
                if AUTO_RELEASE_COLAB:
                    runtime.unassign()


run_cloud()
